# Lab 02 — COPY INTO comparison notebook
## Parameterized Bronze ingestion with `COPY INTO` and a Unity Catalog Volume

This notebook keeps the same Azure, Unity Catalog, schema, validation, and Service Principal setup as the Auto Loader implementation.

The ingestion method is `COPY INTO`, and the source files are read through a Unity Catalog external Volume.

Main characteristics:
- target table is `orders_copy_into`;
- source files are exposed through a UC Volume;
- no Structured Streaming;
- no Auto Loader checkpoint;
- no `cloudFiles`;
- `COPY INTO` tracks files already loaded into the target table and skips them on normal reruns;
- the notebook can be run manually or from a scheduled Databricks Job.

> This notebook is an additional comparison/learning notebook. The Auto Loader notebook remains the primary streaming-style ingestion implementation.


# Lab task mapping

| # | Task | Status / coverage |
|---|---|---|
| 1 | Create personal ADLS container `parvinbadalov` | Completed outside notebook |
| 2 | Create UC external location `parvinbadalov_external_location` | Completed outside notebook |
| 3 | Test external location read/write access | Completed outside notebook |
| 4 | Create Bronze, Silver, Gold schemas backed by external storage | Reused / created if missing |
| 5 | Create personal Azure Key Vault-backed secret scope | Completed outside notebook |
| 6 | Create source/ingestion folder | Completed outside notebook |
| 7 | Upload source files | Completed outside notebook |
| 8 | Build alternative Bronze ingestion notebook | This notebook |
| 9 | Add metadata columns | Yes |
| 10 | Make ingestion idempotent | Yes, through `COPY INTO` file tracking |
| 11 | Write data as Delta table to Bronze schema | Yes |
| 12 | Create Databricks Job | Optional comparison Job |
| 13 | Configure Job to use shared all-purpose cluster | Use `GP1` |
| 14 | Run and validate | Validation controls included |
| 15 | Configure legacy Service Principal access from secret scope | Yes |
| 16 | Create legacy `dbutils.fs.mount` | Not supported on current shared cluster |
| 17 | Test legacy storage access | Yes, via direct `abfss://` access |


# 1. Parameters

The notebook uses Databricks widgets so the same code can run interactively or as a Databricks Job.

For normal Job runs, keep optional validation switches set to `false`.


In [0]:
# Core Azure storage parameters
dbutils.widgets.text("storage_account", "dlspl21databricks")
dbutils.widgets.text("container", "parvinbadalov")

# Unity Catalog names
dbutils.widgets.text("catalog", "dbr_dev")
dbutils.widgets.text("bronze_schema", "parvinbadalov_bronze")
dbutils.widgets.text("silver_schema", "parvinbadalov_silver")
dbutils.widgets.text("gold_schema", "parvinbadalov_gold")
dbutils.widgets.text("volume_name", "raw_landing")
dbutils.widgets.text("target_table", "orders_copy_into")

# ADLS folder names
dbutils.widgets.text("source_dir", "ingestion")
dbutils.widgets.text("bronze_dir", "bronze")
dbutils.widgets.text("silver_dir", "silver")
dbutils.widgets.text("gold_dir", "gold")

# Shared secret scope for legacy SPN access
dbutils.widgets.text("secret_scope", "default2")

# Optional development/testing switches
dbutils.widgets.dropdown("run_validation", "true", ["true", "false"])
dbutils.widgets.dropdown("run_idempotency_test", "false", ["true", "false"])
dbutils.widgets.dropdown("run_spn_access_test", "false", ["true", "false"])


# 2. Build paths and object names

The ABFSS paths are built from the parameter values.

The resulting paths are covered by the Unity Catalog external location `parvinbadalov_external_location`.


In [0]:
storage_account = dbutils.widgets.get("storage_account")
container = dbutils.widgets.get("container")

catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")
gold_schema = dbutils.widgets.get("gold_schema")
volume_name = dbutils.widgets.get("volume_name")
target_table = dbutils.widgets.get("target_table")

source_dir = dbutils.widgets.get("source_dir").strip("/")
bronze_dir = dbutils.widgets.get("bronze_dir").strip("/")
silver_dir = dbutils.widgets.get("silver_dir").strip("/")
gold_dir = dbutils.widgets.get("gold_dir").strip("/")

SECRET_SCOPE = dbutils.widgets.get("secret_scope")

run_validation = dbutils.widgets.get("run_validation").lower() == "true"
run_idempotency_test = dbutils.widgets.get("run_idempotency_test").lower() == "true"
run_spn_access_test = dbutils.widgets.get("run_spn_access_test").lower() == "true"

storage_root = f"abfss://{container}@{storage_account}.dfs.core.windows.net"

# Physical cloud locations
source_cloud_path = f"{storage_root}/{source_dir}/"
bronze_location = f"{storage_root}/{bronze_dir}/"
silver_location = f"{storage_root}/{silver_dir}/"
gold_location = f"{storage_root}/{gold_dir}/"

# Unity Catalog object names
volume_fqn = f"{catalog}.{bronze_schema}.{volume_name}"
volume_path = f"/Volumes/{catalog}/{bronze_schema}/{volume_name}/"
bronze_table = f"{catalog}.{bronze_schema}.{target_table}"

print("Source cloud path:", source_cloud_path)
print("Volume:", volume_fqn)
print("Volume path:", volume_path)
print("Bronze schema location:", bronze_location)
print("Silver schema location:", silver_location)
print("Gold schema location:", gold_location)
print("Target Bronze table:", bronze_table)


# 3. Create externally backed Bronze, Silver, and Gold schemas

These statements are safe to rerun because they use `IF NOT EXISTS`.

The schemas already created by the Auto Loader notebook will simply be reused.


In [0]:
spark.sql(f'''
CREATE SCHEMA IF NOT EXISTS {catalog}.{bronze_schema}
MANAGED LOCATION '{bronze_location}'
''')

spark.sql(f'''
CREATE SCHEMA IF NOT EXISTS {catalog}.{silver_schema}
MANAGED LOCATION '{silver_location}'
''')

spark.sql(f'''
CREATE SCHEMA IF NOT EXISTS {catalog}.{gold_schema}
MANAGED LOCATION '{gold_location}'
''')


In [0]:
# Validate that the personal schemas exist
display(
    spark.sql(
        f"SHOW SCHEMAS IN {catalog} LIKE 'parvinbadalov*'"
    )
)


# Create an external Volume for the landing files

The Volume is created inside the Bronze schema and points to the existing ADLS `ingestion/` folder.

This does **not** duplicate or move the CSV files. It exposes the same cloud files through the Unity Catalog-governed path:

```text
/Volumes/dbr_dev/parvinbadalov_bronze/raw_landing/
```

`COPY INTO` will read from this Volume path instead of reading the ABFSS source path directly.


In [0]:
spark.sql(f'''
CREATE EXTERNAL VOLUME IF NOT EXISTS {volume_fqn}
LOCATION '{source_cloud_path}'
''')

# Validate the Volume and list the landing files
display(spark.sql(f"DESCRIBE VOLUME {volume_fqn}"))
display(dbutils.fs.ls(volume_path))


# 4. Create the COPY INTO target Delta table

Unlike Auto Loader, `COPY INTO` loads into an existing target table.

The table includes:
- the seven source columns;
- source file path;
- ingestion timestamp;
- load date.


In [0]:
spark.sql(f'''
CREATE TABLE IF NOT EXISTS {bronze_table} (
    Order_ID STRING,
    Product STRING,
    Category STRING,
    Quantity INT,
    Price DOUBLE,
    City STRING,
    Date DATE,
    _source_file STRING,
    _ingested_at TIMESTAMP,
    _load_date DATE
)
USING DELTA
''')


# Load new files from the Volume with COPY INTO

`COPY INTO` loads files exposed through the Unity Catalog Volume into the Delta target table.

The Volume path is:

```text
/Volumes/dbr_dev/parvinbadalov_bronze/raw_landing/
```

On normal reruns, files already loaded by the same `COPY INTO` target are skipped automatically.

This notebook intentionally does **not** use `FORCE = TRUE`, because forcing a reload would defeat the idempotency demonstration.


In [0]:
copy_sql = f'''
COPY INTO {bronze_table}
FROM (
    SELECT
        Order_ID,
        Product,
        Category,
        CAST(Quantity AS INT) AS Quantity,
        CAST(Price AS DOUBLE) AS Price,
        City,
        CAST(Date AS DATE) AS Date,
        _metadata.file_path AS _source_file,
        current_timestamp() AS _ingested_at,
        current_date() AS _load_date
    FROM '{volume_path}'
)
FILEFORMAT = CSV
FORMAT_OPTIONS (
    'header' = 'true',
    'dateFormat' = 'yyyy-MM-dd'
)
'''

copy_result = spark.sql(copy_sql)
display(copy_result)


# 6. Optional validation

For interactive development:
- `run_validation = true`

For normal Job runs:
- `run_validation = false`


In [0]:
if run_validation:
    display(spark.table(bronze_table))

    row_count = spark.table(bronze_table).count()
    print("COPY INTO Bronze row count:", row_count)

    required_metadata_columns = {
        "_source_file",
        "_ingested_at",
        "_load_date"
    }

    actual_columns = set(spark.table(bronze_table).columns)
    missing_columns = required_metadata_columns - actual_columns

    assert not missing_columns, f"Missing metadata columns: {missing_columns}"

    print("Metadata validation passed.")


# 7. Optional idempotency test

Set:
- `run_idempotency_test = true`

The notebook runs the same `COPY INTO` statement again.

With no new files:
- the row count before and after must be equal;
- already loaded files must not be duplicated.


In [0]:
if run_idempotency_test:
    count_before = spark.table(bronze_table).count()

    second_copy_result = spark.sql(copy_sql)
    display(second_copy_result)

    count_after = spark.table(bronze_table).count()

    print("Before rerun:", count_before)
    print("After rerun :", count_after)

    assert count_before == count_after, "COPY INTO idempotency test failed"

    print("COPY INTO idempotency test passed.")


# 8. Databricks Job configuration

This notebook can be run manually or from a separate Job/task.

## Recommended task parameters

| Key | Value |
|---|---|
| `storage_account` | `dlspl21databricks` |
| `container` | `parvinbadalov` |
| `catalog` | `dbr_dev` |
| `bronze_schema` | `parvinbadalov_bronze` |
| `silver_schema` | `parvinbadalov_silver` |
| `gold_schema` | `parvinbadalov_gold` |
| `volume_name` | `raw_landing` |
| `target_table` | `orders_copy_into` |
| `source_dir` | `ingestion` |
| `bronze_dir` | `bronze` |
| `silver_dir` | `silver` |
| `gold_dir` | `gold` |
| `secret_scope` | `default2` |
| `run_validation` | `false` |
| `run_idempotency_test` | `false` |
| `run_spn_access_test` | `false` |

## Compute

Use the shared all-purpose cluster:

`GP1`

## Trigger recommendation

Because the file-arrival trigger currently fails with Azure IAM authorization errors on the shared environment, use:
- manual `Run now` for testing, or
- a scheduled trigger.

`COPY INTO` will still skip files already loaded into the target table.


# 9. Legacy Service Principal access from the shared secret scope

The shared scope contains:
- `tenant-id`
- `sp-databricks-adls-appid`
- `sp-databricks-adls-appkey`

The secret values must never be printed.


In [0]:
if run_spn_access_test:
    tenant_id = dbutils.secrets.get(
        scope=SECRET_SCOPE,
        key="tenant-id"
    )

    client_id = dbutils.secrets.get(
        scope=SECRET_SCOPE,
        key="sp-databricks-adls-appid"
    )

    client_secret = dbutils.secrets.get(
        scope=SECRET_SCOPE,
        key="sp-databricks-adls-appkey"
    )

    print("Client ID loaded:", bool(client_id))
    print("Client secret loaded:", bool(client_secret))
    print("Tenant ID loaded:", bool(tenant_id))


# 10. Supervisor-recommended legacy access alternative

The academy Shared access mode cluster blocks `dbutils.fs.mount()`.

When `run_spn_access_test = true`, Spark is configured with Service Principal OAuth settings and direct `abfss://` access is validated.


In [0]:
if run_spn_access_test:
    spark.conf.set(
        f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net",
        "OAuth"
    )

    spark.conf.set(
        f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net",
        "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
    )

    spark.conf.set(
        f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net",
        client_id
    )

    spark.conf.set(
        f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net",
        client_secret
    )

    spark.conf.set(
        f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net",
        f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
    )

    display(dbutils.fs.ls(f"{storage_root}/"))


# 11. Final comparison checklist

Validate and capture:

- external Volume: `dbr_dev.parvinbadalov_bronze.raw_landing`

1. Auto Loader table:
   - `dbr_dev.parvinbadalov_bronze.orders_autoloader`
2. COPY INTO table:
   - `dbr_dev.parvinbadalov_bronze.orders_copy_into`
3. Both tables contain:
   - `_source_file`
   - `_ingested_at`
   - `_load_date`
4. Rerunning Auto Loader without new files does not increase its row count.
5. Rerunning `COPY INTO` without new files does not increase its row count.
6. Adding a new CSV and rerunning each method loads only the new data for that method's own tracking state.

## Important comparison note

Auto Loader and `COPY INTO` keep their ingestion state differently:

- Auto Loader uses the configured checkpoint directory.
- `COPY INTO` keeps file-load history for the target table.

Therefore the two methods should write to different target tables.
